In [9]:
import pandas as pd
import numpy as np

In [15]:
filename = "../data/mooloolaba_wave_data_2015-2025.csv"

df = pd.read_csv(
    filename, 
    parse_dates=['datetime_aest'], 
    index_col='datetime_aest'
)

df.info()

df.describe()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 179404 entries, 2015-01-01 00:00:00 to 2025-03-31 23:30:00
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   hsig_m        179404 non-null  float64
 1   hmax_m        179404 non-null  float64
 2   tz_s          179404 non-null  float64
 3   tp_s          179404 non-null  float64
 4   peak_dir_deg  179404 non-null  float64
 5   sst_c         179404 non-null  float64
dtypes: float64(6)
memory usage: 9.6 MB


,hsig_m,hmax_m,tz_s,tp_s,peak_dir_deg,sst_c
count,179404.000000,179404.000000,179404.000000,179404.000000,179404.000000,179404.000000
mean,0.618247,1.452710,5.022036,8.481946,93.613014,20.773759
std,7.781178,7.877351,8.163323,8.734153,38.475673,18.890006
min,-99.900000,-99.900000,-99.900000,-99.900000,-99.900000,-99.900000
25%,0.841000,1.410000,4.961000,7.277000,85.000000,21.450000
50%,1.106000,1.860000,5.531000,8.937900,99.000000,23.500000
75%,1.486000,2.510000,6.193000,10.843000,115.000000,25.900000
max,5.203600,9.221900,11.003000,21.121000,358.000000,30.000000


In [16]:
df = df.replace(-99.9, np.nan)
df = df.interpolate()

In [17]:
df.describe()

,hsig_m,hmax_m,tz_s,tp_s,peak_dir_deg,sst_c
count,179404.000000,179404.000000,179404.000000,179404.000000,179404.000000,179404.000000
mean,1.218376,2.058745,5.649178,9.132260,98.436479,23.700906
std,0.515764,0.877053,0.981265,2.493247,23.947082,2.432721
min,0.036800,0.107800,3.036000,2.514000,0.000000,-5.000000
25%,0.847000,1.420000,4.974475,7.295000,87.000000,21.600000
50%,1.110000,1.870000,5.541000,8.958000,101.000000,23.700000
75%,1.488000,2.510000,6.200000,10.848000,115.000000,25.900000
max,5.203600,9.221900,11.003000,21.121000,358.000000,30.000000


In [ ]:
## create target variable of next day significant wave height

target_col_name = 'hsig_m_next_day'

df[target_col_name] = df['hsig_m'].shift(-1)
df.dropna(inplace=True)
print(len(df))

features = [e for e in df.columns if e != target_col_name]
print(features)

179403
['hsig_m', 'hmax_m', 'tz_s', 'tp_s', 'peak_dir_deg', 'sst_c']


In [27]:
X = df[features]
y = df[target_col_name]

In [28]:
X

,hsig_m,hmax_m,tz_s,tp_s,peak_dir_deg,sst_c
datetime_aest,,,,,,
2015-01-01 00:00:00,1.1364,2.08,6.6727,9.3271,97.0,26.95
2015-01-01 00:30:00,1.1240,1.92,6.3688,10.1230,92.0,26.90
2015-01-01 01:00:00,1.2728,2.07,6.6257,10.2040,90.0,26.90
2015-01-01 01:30:00,1.2310,1.85,6.6525,9.7295,99.0,26.90
2015-01-01 02:00:00,1.1512,2.00,6.0310,10.3206,84.0,26.90
...,...,...,...,...,...,...
2025-03-31 21:00:00,1.0750,2.07,5.0890,8.5740,105.0,26.25
2025-03-31 21:30:00,1.0940,1.76,4.9720,8.1460,125.0,26.20
2025-03-31 22:00:00,1.1500,1.84,5.2150,8.8260,120.0,26.15


In [29]:
y

datetime_aest
2015-01-01 00:00:00    1.1240
2015-01-01 00:30:00    1.2728
2015-01-01 01:00:00    1.2310
2015-01-01 01:30:00    1.1512
2015-01-01 02:00:00    1.2148
                        ...  
2025-03-31 21:00:00    1.0940
2025-03-31 21:30:00    1.1500
2025-03-31 22:00:00    1.2090
2025-03-31 22:30:00    1.2310
2025-03-31 23:00:00    1.2570
Name: hsig_m_next_day, Length: 179403, dtype: float64

In [30]:
split_index = int(len(X)* 0.8)

X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

print(len(X_train))
print(len(X_test))
print(len(y_train))
print(len(y_test))

143522
35881
143522
35881


In [26]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [31]:
# Initialize the models
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=42)
gb = GradientBoostingRegressor(n_estimators=100, random_state=42)

# Train the models
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
gb.fit(X_train, y_train)

,loss,'squared_error'
,learning_rate,0.1
,n_estimators,100
,subsample,1.0
,criterion,'friedman_mse'
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_depth,3
,min_impurity_decrease,0.0
,init,None


In [32]:
# Make predictions
lr_preds = lr.predict(X_test)
rf_preds = rf.predict(X_test)
gb_preds = gb.predict(X_test)

# Evaluate the models
def evaluate_model(predictions, actual):
  mae = mean_absolute_error(actual, predictions)
  rmse = np.sqrt(mean_squared_error(actual, predictions))
  print(f"MAE: {mae:.4f}")
  print(f"RMSE: {rmse:.4f}")

print("Linear Regression:")
evaluate_model(lr_preds, y_test)
print("\nRandom Forest:")
evaluate_model(rf_preds, y_test)
print("\nGradient Boosting:")
evaluate_model(gb_preds, y_test)

Linear Regression:
MAE: 0.0562
RMSE: 0.0793

Random Forest:
MAE: 0.0577
RMSE: 0.0814

Gradient Boosting:
MAE: 0.0563
RMSE: 0.0794
